In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable
from datetime import datetime
import uuid
from datetime import datetime, UTC
datetime.now(UTC)

In [0]:
spark.sql("use catalog novacart_adb")
spark.sql("create schema if not exists gold_schema")
gold_run_id = str(uuid.uuid4())

run_ts_str =datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

run_date_str = datetime.utcnow().strftime("%Y-%m-%d")

print("current gold run id:", gold_run_id)
print("run timestamp folder ", run_ts_str)


Gold Control Table

In [0]:
spark.sql("""
          create table if not exists novacart_adb.gold_schema.processing_control(
            layer string,
            entity_name string,
            last_processed_silver_run_id string,
            last_processed_silver_run_ts timestamp,
            rows_merged bigint,
            run_status string,
            gold_run_id string,
            updated_at timestamp
          ) using delta
          """)

Helper functions

In [0]:
def upsert_to_gold(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark,target_table)
        (dt.alias("target")
         .merge(df_source.alias("source"),f"target.{target_table} = source.{join_key}")
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(entity_name:str):
    ctrl = (
        spark.table("novacart_adb.gold_schema.processing_control")
             .filter(
                 (f.col("layer") == "gold") &
                 (f.col("entity_name") == entity_name) &
                 (f.col("run_status") == "success")
             )
             .orderBy(f.col("updated_at").desc())
             .limit(1)
             )
    
    rows = ctrl.collect()
    if not rows:
        return None

    return rows[0]["last_processed_silver_run_ts"]    


In [0]:
def upsert_gold_control(entity_name,last_processed_silver_run_id,last_processed_run_ts,rows_merged):
    ctrl_df = spark.createDataFrame(
        [(
            "gold",
            entity_name,
            last_processed_silver_run_id,
            last_processed_run_ts,
            int(rows_merged),
            "success",
            gold_run_id,
            datetime.utcnow()
        )],
        schema="""
            layer string,
            entity_name string,
            last_processed_silver_run_id string,
            last_processed_silver_run_ts timestamp,
            rows_merged bigint,
            run_status string,
            gold_run_id string,
            updated_at timestamp
            """
    )

    dt = DeltaTable.forName(spark,"novacart_adb.gold_schema.processing_control")
    (dt.alias("t")
     .merge(ctrl_df.alias("s"),"t.layer = s.layer and t.entity_name = s.entity_name")   
     .whenMatchedUpdate(set={
         "last_processed_silver_run_id": "s.last_processed_silver_run_id",
         "last_processed_silver_run_ts" : "s.last_processed_silver_run_ts",
         "rows_merged" : "s.rows_merged",
         "run_status" : "s.run_status",
         "gold_run_id" : "s.gold_run_id",
         "updated_at" : "s.updated_at"
     })
     .whenNotMatchedInsertAll()
     .execute())
     

     

Read Changed silver rows only

In [0]:
last_gold_ts = get_last_processed_silver_ts("orders_information")

print(f"last processed silver timestamp for gold = {last_gold_ts}")

silver_orders_current = spark.read.table("novacart_adb.silver_schema.orders_transformed")
silver_products_current = spark.read.table("novacart_adb.silver_schema.products_transformed")
silver_payments_current = spark.read.table("novacart_adb.silver_schema.payments_transformed")

if last_gold_ts is None:
  changed_orders = silver_orders_current
  changed_products = silver_products_current
  changed_payments = silver_payments_current
else:
  changed_orders = silver_orders_current.filter(f.col("updated_at") > f.lit(last_gold_ts))
  changed_products = silver_products_current.filter(f.col("updated_at") > f.lit(last_gold_ts))
  changed_payments = silver_payments_current.filter(f.col("updated_at") > f.lit(last_gold_ts))

changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print(f"Number of changed orders =  {changed_orders_count}")
print(f"Number of changed products = {changed_products_count}")
print(f"Number of changed payments = {changed_payments_count}")


                                                


In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()
impacted_from_products = (
    changed_products.alias("p").join(changed_orders.alias("o"),f.col("p.product_id") == f.col("o.product_id"),"inner")).select(f.col("o.order_id")).distinct()

impacted_orders_ids = (
    impacted_from_orders
    .union(impacted_from_payments)
    .union(impacted_from_products)
    .distinct()
)
print("impacted_orders_ids = ", impacted_orders_ids.count())
display(impacted_orders_ids.orderBy("order_id")
        )

In [0]:
impacted_orders = (
    silver_orders_current.alias("o")
    .join(impacted_orders_ids.alias("i"),f.col("o.order_id") == f.col("i.order_id"),"inner")
    .select("o.*")
)
gold_delta = (
    impacted_orders.alias("o")
    .join(
        silver_products_current.alias("p"),
        f.col("o.product_id") == f.col("p.product_id"),
        "inner"
    )
    .join(
        silver_payments_current.alias("py"),
        f.col("o.order_id") == f.col("py.order_id"),
        "inner"
    )
    .select(
        f.col("o.order_id"),
        f.col("o.customer_id"),
        f.col("p.product_id"),
        f.col("p.product_name"),
        f.col("p.category"),
        f.col("p.price").alias("products_price"),
        f.col("o.order_status"),
        f.col("o.order_amount"),
        f.col("py.payment_id"),
        f.col("py.payment_status"),
        f.col("py.paid_amount"),
        f.col("o.order_date"),
        f.col("o.order_month"),
        f.col("o.order_year"),
        f.greatest(
            f.col("o.updated_at").cast("timestamp"),
            f.col("p.updated_at").cast("timestamp"),
            f.col("py.processed_at").cast("timestamp")
            ).alias("gold_updated_ts")
    )
    .dropDuplicates(["order_id"])
    .withColumn(
        "payment_completion_ratio",
        f.when(
            f.col("order_amount") > 0,
            f.col("paid_amount")/f.col("order_amount"))
        .otherwise(f.lit(0.0))
    )
    .withColumn(
        "payment_state",
            f.when(f.col("order_amount") == 0, "Invalid_amount")
            .when(f.col("payment_completion_ratio") == 0, "Unpaid")
            .when(f.col("payment_completion_ratio") == 1, "Paid")
            .when(f.col("payment_completion_ratio") < 1, "Partially_paid")
            .when(f.col("payment_completion_ratio") > 1, "Overpaid")
    )
    .withColumn("gold_updated_date",f.to_date("gold_updated_ts"))
    .withColumn("gold_run_id",f.lit(gold_run_id))
        

)
print("gold_delta_rows = ", gold_delta.count())
display(gold_delta)
        


In [0]:
if gold_delta.count() > 0:
    upsert_to_gold(gold_delta,"novacart_adb.gold_schema.orders_information","order_id")
else:
    print("No new rows to insert in gold table")

In [0]:
%sql
select * from novacart_adb.gold_schema.orders_information

In [0]:
if not spark.catalog.tableExists("novacart_adb.gold_schema.orders_information_scd2"):
    spark.sql("""
              create table novacart_adb.gold_schema.orders_information_scd2
              using delta 
              as
              select *,
                  cast(null as timestamp) as valid_from_ts,
                  cast(null as timestamp) as valid_to_ts,
                  true as is_current
                  from novacart_adb.gold_schema.orders_information
                  where 1 = 0
              """
              )
    
if gold_delta.count() > 0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    spark.sql("""
        MERGE INTO novacart_adb.gold_schema.orders_information_scd2 t
        using gold_delta_view as s
        ON t.order_id = s.order_id AND t.is_current = true
        WHEN MATCHED AND (
            not (t.order_status <=> s.order_status) or
            not (t.order_amount <=> s.order_amount) or
            not (t.paid_amount <=> s.paid_amount) or
            not (t.product_id  <=> s.product_id) or
            not (t.category <=> s.category) or
            not (t.product_name <=> s.product_name) or
            not (t.products_price <=> s.products_price))
        THEN UPDATE SET 
            is_current = false,
            valid_to_ts = s.gold_updated_ts
        """)

spark.sql("""
         insert into novacart_adb.gold_schema.orders_information_scd2 
             select 
                    s.order_id,
                    s.customer_id,
                    s.product_id,
                    s.product_name,
                    s.category,
                    s.products_price,
                    s.order_status,
                    s.order_amount,
                    s.payment_status,
                    s.paid_amount,
                    s.order_date,
                    s.order_month,
                    s.order_year,
                    s.gold_updated_ts,
                    s.payment_completion_ratio,
                    s.payment_state,
                    s.gold_updated_date,
                    s.gold_run_id,
                    s.gold_updated_ts as valid_from_ts,
                    cast(null as timestamp) as valid_to_ts,
                    true as is_current
             from gold_delta_view s
             left join novacart_adb.gold_schema.orders_information_scd2 t
             on s.order_id = t.order_id and t.is_current = true 
             where t.order_id is null or (
                 not (t.order_status <=> s.order_status) or
                 not (t.order_amount <=> s.order_amount) or
                 not (t.paid_amount <=> s.paid_amount) or
                 not (t.product_id <=> s.product_id) or
                 not (t.category <=> s.category) or
                 not (t.product_name <=> s.product_name) or
                 not (t.products_price <=> s.products_price))
             """)    


     

update category-level gold aggregation

In [0]:
if gold_delta.count() > 0:
    impacted_categories = (
        gold_delta
        .select("category")
        .filter(f.col("category").isNotNull())
        .distinct())
    
category_perf_delta = (
    spark.read.table("novacart_adb.gold_schema.orders_information")
    .join(impacted_categories,"category","inner")
    .groupBy("category")
    .agg(
        f.countDistinct("order_id").alias("total_orders"),
        f.sum(
            f.when(f.col("order_amount") > 0, f.col("order_amount"))
            .otherwise(f.lit(0.0)))
            .alias("Gross_merchandise_value"),
        f.sum(
            f.when(f.col("paid_amount") > 0,f.col("paid_amount"))
            .otherwise(f.lit(0.0)))
            .alias("total_paid_amount"),
        f.avg(f.col("payment_completion_ratio")).alias("Average_payment_completion_ratio"),
        (
            f.sum(f.when(f.col("payment_status") == "FAILED", 1).otherwise(0))/f.count("*")
        ).alias("payment_failure_rate")          
    )
)
upsert_to_gold(category_perf_delta,"novacart_adb.gold_schema.category_performance","category")

In [0]:
%sql
select * from novacart_adb.gold_schema.category_performance

Publish Gold snapshots to Volume

In [0]:
spark.sql("create volume if not exists novacart_adb.gold_schema.gold_snapshot_vol")

In [0]:
latest_orders_path = (
    "/Volumes/novacart_adb/gold_schema/gold_snapshot_vol/gold_latest/orders_information"
)

latest_category_path = (
    "/Volumes/novacart_adb/gold_schema/gold_snapshot_vol/gold_latest/category_performance"
)

historical_orders_path = (
    f"/Volumes/novacart_adb/gold_schema/gold_snapshot_vol/gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str}"
)
historical_category_path = (
    f"/Volumes/novacart_adb/gold_schema/gold_snapshot_vol/gold_snapshots/category_performance/run_date={run_date_str}/run_ts={run_ts_str}"
)

spark.read.table("novacart_adb.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(latest_orders_path)
spark.read.table("novacart_adb.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(latest_category_path)
spark.read.table("novacart_adb.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(historical_orders_path)
spark.read.table("novacart_adb.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(historical_category_path)

print("latest orders path :", latest_orders_path)
print("latest category path :", latest_category_path)
print("historical orders path :", historical_orders_path)
print("historical category path:", historical_category_path)

Update Gold control table

In [0]:
latest_silver_ts = silver_orders_current.agg(f.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]

latest_silver_run_id = (
    silver_orders_current
    .filter(f.col("bronze_ingested_at") == latest_silver_ts)
    .agg(f.max("silver_run_id").alias("mx"))
    .collect()[0]["mx"]
)   if latest_silver_ts is not None else None

upsert_gold_control("orders_information",latest_silver_run_id,latest_silver_ts,gold_delta.count())
display(spark.table("novacart_adb.gold_schema.processing_control"))

